# Public peer review document histories

Run all cells to reproduce the numerical results from the public GitHub repository.


In [ ]:
from pathlib import Path
import hashlib
import io
import json
import os
import re
import tempfile
import urllib.request
import zipfile

repository = "evidenceworks/peer-review-document-histories"
explicit = os.environ.get("PEER_REVIEW_REPOSITORY")
locations = [Path(explicit)] if explicit else [Path.cwd(), Path.cwd().parent]
package = next((p.resolve() for p in locations
                if (p / "scripts/reproduce.py").is_file()
                and (p / "data/manifest.sha256").is_file()), None)

if package is None:
    headers = {"User-Agent": "peer-review-document-histories",
               "Accept": "application/vnd.github+json"}
    commit_request = urllib.request.Request(
        f"https://api.github.com/repos/{repository}/commits/main", headers=headers)
    with urllib.request.urlopen(commit_request, timeout=30) as response:
        revision = json.load(response)["sha"]
    if not re.fullmatch(r"[0-9a-f]{40}", revision):
        raise ValueError("GitHub returned an invalid commit identifier.")

    archive_request = urllib.request.Request(
        f"https://codeload.github.com/{repository}/zip/{revision}", headers=headers)
    with urllib.request.urlopen(archive_request, timeout=60) as response:
        contents = response.read(100_000_001)
    if len(contents) > 100_000_000:
        raise ValueError("The repository archive exceeds the expected size.")

    destination = Path(tempfile.mkdtemp(prefix="document_histories_")).resolve()
    with zipfile.ZipFile(io.BytesIO(contents)) as archive:
        if sum(item.file_size for item in archive.infolist()) > 250_000_000:
            raise ValueError("The repository archive exceeds the expected size.")
        if any(not (destination / item.filename).resolve().is_relative_to(destination)
               for item in archive.infolist()):
            raise ValueError("Invalid path in repository archive.")
        archive.extractall(destination)
    matches = [p.parent.parent for p in destination.rglob("scripts/reproduce.py")
               if (p.parent.parent / "data/manifest.sha256").is_file()]
    if len(matches) != 1:
        raise ValueError("The repository archive does not contain one matching dataset and analysis.")
    package = matches[0]
    print(f"Source: {repository} ({revision})")
else:
    print(f"Source: {package}")

expected = {
    "data/manifest.sha256": "c4f5f6b8062355b52d865bee0b04448fcad566f88ba1440c2078a26ca32b7e53",
    "scripts/reproduce.py": "2cc1b430329079c1ff72a4e03111371bf7bb8b0b6df9e3c6dde61ad1bb2ab29a",
}
for relative, digest in expected.items():
    if hashlib.sha256((package / relative).read_bytes()).hexdigest() != digest:
        raise ValueError(f"Repository files do not match this notebook: {relative}")
print("Data manifest and analysis code verified.")


## Analysis

Reconstruct source-defined histories, conditional intervals and exact-text comparisons. The script verifies all distributed input files before calculation.


In [ ]:
import subprocess
import sys

results = package / "results"
subprocess.run([sys.executable, str(package / "scripts/reproduce.py"),
                "--output", str(results)], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(package / "tests"), "-v"], check=True)


In [ ]:
result_archive = package / "document_history_results.zip"
output_names = ("statistics.json", "family_outcomes.csv", "step_coverage.csv", "conditional_estimates.csv")
with zipfile.ZipFile(result_archive, "w", zipfile.ZIP_DEFLATED) as archive:
    for name in output_names:
        archive.write(results / name, name)
try:
    from google.colab import files
except ImportError:
    print("Numerical results:", result_archive)
else:
    files.download(str(result_archive))
